# Parking Space Classification

## UA2 Project — Applied Deep Learning

This project compares a custom convolutional neural network (CNN) with two data augmentation strategies and a pretrained ResNet18 model using feature extraction and fine-tuning.

**Classes**
- `0` — Occupied
- `1` — Available
- `2` — Available but restricted

The notebook covers image loading and preprocessing, a fixed train/validation/test split, model training, confusion matrices, a performance comparison, and examples of misclassified test images.

**Hardware:** CUDA is used when an NVIDIA GPU is available. Otherwise, the notebook runs on CPU.

## 0. Imports, Reproducibility, and Device Check

In [ ]:
import os, random, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Appareil :", device)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("GPU non détecté : sélectionner un serveur/noyau CoCalc avec GPU NVIDIA.")

## 1. Dataset Location

Place the processed image dataset in `data/images_128`, using this folder structure:

```text
data/
└── images_128/
    ├── 0/
    ├── 1/
    └── 2/
```

The folder names determine the class indices. See the repository README for setup instructions.

## 2. Import Libraries and Configure the Environment

In [ ]:
# General libraries
import os
import random
import time
import copy

# Data manipulation
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Dataset splitting
from sklearn.model_selection import train_test_split

# Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configure GPU or CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch :", torch.__version__)
print("Appareil utilisé :", DEVICE)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("GPU non détecté")

## 3. Load and Verify Images

In [ ]:
# Load images from the three class folders
dataset = datasets.ImageFolder(DATA_DIR)

# Display the folder-to-class mapping
print("Correspondance des classes :", dataset.class_to_idx)

# Display the total number of images
print("Nombre total d'images :", len(dataset))

# Count images in each class
for class_name, class_index in dataset.class_to_idx.items():
    count = sum(1 for _, label in dataset.samples if label == class_index)
    print(f"Classe {class_name} : {count} images")

- **ImageFolder:** a PyTorch utility that automatically loads images organized into class-specific folders.

## 4. Visualize Sample Images

In [ ]:
# Class names for visualization
class_labels = {
    0: "Occupe",
    1: "Available",
    2: "Available but restricted"
}

# Display three random images per class
fig, axes = plt.subplots(3, 3, figsize=(10, 9))

for class_index in range(3):

    # Get indices for the current class
    indices = [
        i for i, label in enumerate(dataset.targets)
        if label == class_index
    ]

    # Select three random images
    selected_indices = random.sample(indices, 3)

    for j, idx in enumerate(selected_indices):
        image, label = dataset[idx]

        axes[class_index, j].imshow(image)
        axes[class_index, j].set_title(class_labels[label])
        axes[class_index, j].axis("off")

plt.tight_layout()
plt.show()

## 5. Resize and Preprocess Images

In [ ]:
# Define the base image transformations
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# Apply transformations to the dataset
dataset.transform = transform

# Check the shape of one image
image, label = dataset[0]

print("Dimensions de l'image :", image.shape)
print("Étiquette :", label)

- **Resize:** makes all images the same dimensions.

- **128 × 128 pixels:** reduces computational cost while retaining useful visual details.

- **ToTensor:** converts each image into a PyTorch tensor and scales pixel values to the `[0, 1]` range.

## 6. Detect and Exclude Corrupted Images

In [ ]:
from PIL import Image

valid_samples = []
corrupted_images = []

for path, label in dataset.samples:
    try:
        with Image.open(path) as img:
            img.verify()
        valid_samples.append((path, label))

    except Exception:
        corrupted_images.append(path)

# Update the dataset without corrupted images
dataset.samples = valid_samples
dataset.imgs = valid_samples
dataset.targets = [label for _, label in valid_samples]

print("Images valides :", len(dataset))
print("Images corrompues :", len(corrupted_images))

for path in corrupted_images:
    print("Fichier problématique :", path)

## 7. Split the Dataset into Training, Validation, and Test Sets

In [ ]:
# Get image indices and labels
indices = np.arange(len(dataset))
labels = np.array(dataset.targets)

# Split: 70% training and 30% temporary
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

# Split the remaining 30% into validation and test sets
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=labels[temp_idx]
)

# Create dataset subsets
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

# Display split sizes
print("Images d'entraînement :", len(train_dataset))
print("Images de validation :", len(val_dataset))
print("Images de test :", len(test_dataset))

- **70% training:** 814 images are used to teach the models to recognize the three parking-space categories.

- **15% validation:** 175 images are used to monitor performance during training and identify overfitting.

- **15% test:** 175 images are held out for the final model evaluation.

- **Stratification:** preserves approximately the same class proportions in each split.

- **Random state:** makes the split reproducible.

- **Subset:** creates the three subsets from the original dataset without duplicating images.

## 8. Create the DataLoaders

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

- **DataLoader:** a PyTorch utility that provides images to the neural network in batches.

- **Batch size = 32:** the network processes 32 images at a time, balancing memory usage and training speed.

- Only the training data are shuffled. Validation and test order remains fixed to make evaluation consistent.

## 9. Inspect a Data Batch

In [ ]:
# Retrieve one batch of training images
images, labels = next(iter(train_loader))

# Display tensor dimensions
print("Dimensions des images :", images.shape)
print("Dimensions des étiquettes :", labels.shape)

# Display class labels in the batch
print("Étiquettes :", labels)

## 10. Build the Baseline CNN

In [ ]:
class ParkingCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(ParkingCNN, self).__init__()

        self.features = nn.Sequential(
            # First convolutional block
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second convolutional block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Third convolutional block
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Initialize the model on GPU or CPU
model = ParkingCNN(num_classes=NUM_CLASSES).to(DEVICE)

print(model)

- **CNN built from scratch:** this model has no pretrained weights. It learns visual features directly from the parking-space images.

- **Convolutional layers (3 → 32 → 64 → 128):** filters detect visual patterns such as edges, lines, textures, vehicles, and signs.

- **ReLU:** introduces non-linearity so the network can learn more complex patterns.

- **MaxPool2d:** reduces feature-map dimensions and computation while retaining strong activations.

- **Flatten:** converts the feature maps into a one-dimensional vector for the classifier.

- **Dropout (0.5):** randomly disables some neurons during training to help reduce overfitting.

- **Output layer (3 units):** produces a score for each parking-space category.

## 11. Configure the Loss Function and Optimizer

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Number of epochs
EPOCHS = 20

- **CrossEntropyLoss:** measures the difference between the model predictions and the true class labels. It is suitable for this three-class classification task.

- **Adam:** updates the model weights to reduce the loss.

- **Learning rate = 0.001:** controls the size of each weight update.

- **20 epochs:** the model makes up to 20 passes through the training dataset.

## 12. Define the Training Function

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs,
                checkpoint_path="best_cnn_model.pth"):

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

    print(f"\nBest validation accuracy : {best_val_acc:.4f}")

    return history

- **`train_model()`:** trains a model, tracks its performance, and saves the checkpoint with the best validation accuracy.

- **`model.train()`:** enables training behavior, including Dropout.

- **`optimizer.zero_grad()`:** clears gradients before the next update.

- **Forward pass:** sends a batch of images through the network to obtain predictions.

- **`loss.backward()`:** computes gradients for trainable parameters.

- **`optimizer.step()`:** updates the trainable weights.

- **`model.eval()` and `torch.no_grad()`:** evaluate the model without updating its weights.

- The function records training and validation loss and accuracy for later analysis.

## 13. Train the Baseline CNN

In [ ]:
# Measure training time
start_time = time.time()

# Train the model
history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    EPOCHS
)

# Calculate elapsed time
training_time = time.time() - start_time

print(f"\nTraining time : {training_time:.2f} secondes")

## Baseline CNN — Training Results

- **Training loss:** decreases from 1.0902 to 0.1772, showing that the model learns the training images.

- **Final training accuracy:** reaches 93.61%.

- **Best validation accuracy:** 69.71% at epoch 16.

- **Overfitting:** the final training accuracy (93.61%) is much higher than the final validation accuracy (67.43%).

- **Training time:** 20.31 seconds for 20 epochs.

**Conclusion:** the baseline CNN learns useful visual features but shows overfitting. Data augmentation is tested next to improve generalization.

## 14. Plot Training and Validation Curves

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history["train_loss"], label="Training")
axes[0].plot(history["val_loss"], label="Validation")
axes[0].set_title("Loss over epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid()

# Accuracy curve
axes[1].plot(history["train_acc"], label="Training")
axes[1].plot(history["val_acc"], label="Validation")
axes[1].set_title("Accuracy over epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

## Baseline CNN — Learning Curves

- **Training loss:** decreases throughout training, indicating that the model fits the training data.

- **Validation loss:** begins to rise after the early epochs, which can indicate overfitting.

- **Training accuracy:** improves substantially.

- **Validation accuracy:** improves more slowly and remains lower than training accuracy.

**Conclusion:** the gap between training and validation performance suggests that the model does not generalize as well to unseen images.

## 15. Evaluate the Baseline CNN on the Test Set

In [ ]:
# Load the best saved weights
model.load_state_dict(
    torch.load("best_cnn_model.pth", map_location=DEVICE)
)

# Set evaluation mode
model.eval()

y_true = []
y_pred = []

# Generate predictions on the test set
with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

class_names = ["Occupied", "Available", "Available but restricted"]

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix — Baseline CNN (Test)")
plt.show()

# Classification report
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

## Baseline CNN — Test Results

- **Overall accuracy:** 74.29% (130 correct predictions out of 175 test images).

- **Occupied spaces:** strongest class, with 89% recall. The model correctly identifies 49 of 55 occupied spaces.

- **Available spaces:** most difficult class, with 56% recall. The model often confuses available spaces with restricted spaces.

- **Available but restricted spaces:** 80% recall. Some examples are still confused with the available class.

**Conclusion:** the baseline CNN reaches 74.29% test accuracy. Its main weakness is distinguishing available spaces from available but restricted spaces.

## 16. Data Augmentation Strategy 1 — Rotation and Color Variation

In [ ]:
# First augmentation strategy
transform_aug1 = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(8),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor()
])

- **Resize (128 × 128):** retains useful details such as vehicles, parking lines, and signs while limiting computational cost.

- **RandomRotation(8):** applies rotations of up to 8 degrees to simulate small camera-angle variations without excessively distorting the scene.

- **ColorJitter:** varies image appearance so the CNN is less dependent on the exact lighting and colors.

- **Brightness = 0.2:** simulates brighter or darker conditions.

- **Contrast = 0.2:** varies the difference between objects and their background.

- **Saturation = 0.2:** varies color intensity so the model does not rely only on color cues.

**Goal:** expose the CNN to realistic visual variations and improve its ability to generalize.

## 17. Create DataLoaders — Augmentation Strategy 1

In [ ]:
from torch.utils.data import Dataset

class ParkingDataset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        original_idx = self.indices[idx]
        path, label = self.base_dataset.samples[original_idx]

        image = self.base_dataset.loader(path)
        image = self.transform(image)

        return image, label


# Training dataset with augmentation
train_dataset_aug1 = ParkingDataset(
    dataset, train_idx, transform_aug1
)

# Keep validation and test data unaugmented
val_dataset_aug1 = ParkingDataset(
    dataset, val_idx, transform
)

test_dataset_aug1 = ParkingDataset(
    dataset, test_idx, transform
)

# Create DataLoaders
train_loader_aug1 = DataLoader(
    train_dataset_aug1, batch_size=BATCH_SIZE, shuffle=True
)

val_loader_aug1 = DataLoader(
    val_dataset_aug1, batch_size=BATCH_SIZE, shuffle=False
)

test_loader_aug1 = DataLoader(
    test_dataset_aug1, batch_size=BATCH_SIZE, shuffle=False
)

print("Training :", len(train_dataset_aug1))
print("Validation :", len(val_dataset_aug1))
print("Test :", len(test_dataset_aug1))

A custom dataset applies augmentation only to training images. Validation and test images remain unchanged. DataLoaders then provide the images in batches of 32.

## 18. Train the CNN — Augmentation Strategy 1

In [ ]:
# Create a new CNN with the same architecture
model_aug1 = ParkingCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Define the loss function and optimizer
criterion_aug1 = nn.CrossEntropyLoss()

optimizer_aug1 = torch.optim.Adam(
    model_aug1.parameters(),
    lr=0.001
)

# Measure training time
start_time = time.time()

# Train the model with augmentation
history_aug1 = train_model(
    model_aug1,
    train_loader_aug1,
    val_loader_aug1,
    criterion_aug1,
    optimizer_aug1,
    EPOCHS,
    checkpoint_path="best_cnn_aug1.pth"
)

training_time_aug1 = time.time() - start_time

print(f"Training time : {training_time_aug1:.2f} secondes")

## Augmentation Strategy 1 — Training Results

- **Strategy:** geometric and color augmentation using `RandomRotation(8)` and `ColorJitter` (brightness, contrast, and saturation).

- **Best validation accuracy:** 73.14%.

- **Training time:** 47.15 seconds.

**Conclusion:** this strategy introduces realistic angle and lighting variations. Its test accuracy is 71.43%, so it does not outperform the baseline CNN on the held-out test set.

In [ ]:
# Load the best weights
model_aug1.load_state_dict(
    torch.load("best_cnn_aug1.pth", map_location=DEVICE)
)

model_aug1.eval()

y_true_aug1 = []
y_pred_aug1 = []

with torch.no_grad():
    for images, labels in test_loader_aug1:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model_aug1(images)
        predictions = outputs.argmax(dim=1)

        y_true_aug1.extend(labels.cpu().numpy())
        y_pred_aug1.extend(predictions.cpu().numpy())

# Confusion matrix
cm_aug1 = confusion_matrix(y_true_aug1, y_pred_aug1)

class_names = ["Occupied", "Available", "Available but restricted"]

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_aug1,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("CNN avec augmentation 1 - Test")
plt.show()

print(classification_report(
    y_true_aug1,
    y_pred_aug1,
    target_names=class_names
))

## Augmentation Strategy 1 — Test Results

- **Overall accuracy:** 71.43% (125 correct predictions out of 175 test images).

- **Occupied spaces:** 87% recall. The model generally recognizes spaces containing a vehicle.

- **Available spaces:** 75% recall.

- **Available but restricted spaces:** 53% recall, making this the most difficult class.

- **Main confusion:** 24 restricted spaces are predicted as available.

**Conclusion:** the model recognizes occupied and available spaces reasonably well, but it struggles to distinguish available spaces from restricted spaces.

## 20. Data Augmentation Strategy 2 — Geometric Transformations

In [ ]:
# Second augmentation strategy
transform_aug2 = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor()
])

## Augmentation Strategy 2 — Geometric Transformations

- **Strategy:** geometric augmentation using `RandomAffine`.

- **Rotation:** rotates images by up to 5 degrees to simulate small camera-angle changes.

- **Translation:** shifts the image horizontally and vertically by up to 5%.

- **Scale:** slightly zooms in or out, from 95% to 105%.

- **Resize:** all images are resized to 128 × 128 pixels.

- **Why only 5% translation?** A moderate shift changes the position of objects without excessively distorting the scene. Larger shifts could crop important details, such as vehicles, signs, or parking markings, or create unrealistic examples.

**Goal:** help the CNN recognize parking spaces despite small changes in angle, position, and scale.

## 21. Create DataLoaders — Augmentation Strategy 2

In [ ]:
# Create datasets for augmentation strategy 2

train_dataset_aug2 = ParkingDataset(dataset, train_idx, transform_aug2)
val_dataset_aug2 = ParkingDataset(dataset, val_idx, transform)
test_dataset_aug2 = ParkingDataset(dataset, test_idx, transform)

train_loader_aug2 = DataLoader(
    train_dataset_aug2, batch_size=BATCH_SIZE, shuffle=True
)

val_loader_aug2 = DataLoader(
    val_dataset_aug2, batch_size=BATCH_SIZE, shuffle=False
)

test_loader_aug2 = DataLoader(
    test_dataset_aug2, batch_size=BATCH_SIZE, shuffle=False
)

print("DataLoaders de la stratégie 2 créés.")

## 22. Train the CNN — Augmentation Strategy 2

In [ ]:
# Train the CNN with augmentation strategy 2

model_aug2 = ParkingCNN(num_classes=NUM_CLASSES).to(DEVICE)

criterion_aug2 = nn.CrossEntropyLoss()
optimizer_aug2 = torch.optim.Adam(model_aug2.parameters(), lr=0.001)

start_time = time.time()

history_aug2 = train_model(
    model_aug2,
    train_loader_aug2,
    val_loader_aug2,
    criterion_aug2,
    optimizer_aug2,
    EPOCHS,
    checkpoint_path="best_cnn_aug2.pth"
)

training_time_aug2 = time.time() - start_time

print(f"Training time : {training_time_aug2:.2f} secondes")
print(f"Best validation accuracy : {max(history_aug2['val_acc']):.4f}")

## Augmentation Strategy 2 — Training Results

- **Best validation accuracy:** 74.29%, reached at epoch 19.

- **Final training accuracy:** 79.12%.

- **Final validation accuracy:** 72.57%.

- **Training loss:** decreases from 1.1384 to 0.4903, indicating that the model learns progressively.

- **Generalization:** the final training/validation accuracy gap is 6.55 percentage points, suggesting a relatively limited gap between the two sets.

- **Training time:** 30.44 seconds for 20 epochs.

**Conclusion:** the model learns progressively and reaches 74.29% best validation accuracy.

## 23. Confusion Matrix — CNN with Augmentation Strategy 2 (Test)

In [ ]:
model_aug2.load_state_dict(
    torch.load("best_cnn_aug2.pth", map_location=DEVICE)
)

model_aug2.eval()

y_true_aug2 = []
y_pred_aug2 = []

with torch.no_grad():
    for images, labels in test_loader_aug2:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model_aug2(images)
        predictions = outputs.argmax(dim=1)

        y_true_aug2.extend(labels.cpu().numpy())
        y_pred_aug2.extend(predictions.cpu().numpy())

cm_aug2 = confusion_matrix(y_true_aug2, y_pred_aug2)

class_names = ["Occupied", "Available", "Available but restricted"]

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_aug2,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("CNN avec augmentation 2 - Test")
plt.show()

print(classification_report(
    y_true_aug2,
    y_pred_aug2,
    target_names=class_names
))

## Augmentation Strategy 2 — Test Results

- **Overall accuracy:** 77.14% (135 correct predictions out of 175 test images).

- **Occupied spaces:** 89% recall and 94% precision.

- **Available spaces:** 79% recall.

- **Available but restricted spaces:** 64% recall.

- **Main confusion:** 19 restricted spaces are predicted as available.

**Conclusion:** augmentation strategy 2 improves test accuracy over the baseline CNN and strategy 1, although confusion between the two unoccupied categories remains.

## 24. Prepare DataLoaders for Pretrained ResNet18

In [ ]:
# Prepare images for ResNet18

from torchvision import models

transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset_resnet = ParkingDataset(dataset, train_idx, transform_resnet)
val_dataset_resnet = ParkingDataset(dataset, val_idx, transform_resnet)
test_dataset_resnet = ParkingDataset(dataset, test_idx, transform_resnet)

train_loader_resnet = DataLoader(
    train_dataset_resnet, batch_size=BATCH_SIZE, shuffle=True
)

val_loader_resnet = DataLoader(
    val_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False
)

test_loader_resnet = DataLoader(
    test_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False
)

print("DataLoaders ResNet18 créés.")

## ResNet18 — Image Preprocessing and DataLoaders

- **Resize:** images are resized to 224 × 224 pixels, the input size used in this experiment for ResNet18.

- **Tensor conversion:** converts images into PyTorch tensors.

- **Normalization:** uses the ImageNet channel means and standard deviations expected by the pretrained weights.

- **Dataset splits:** reuses the same training, validation, and test indices to make model comparisons fair.

- **DataLoaders:** load images in batches of 32. Only the training loader shuffles the data.

**Goal:** prepare the images for ResNet18 and reuse visual features learned from ImageNet.

## 25. Transfer Learning — ResNet18 Feature Extraction

In [ ]:
# Load pretrained ResNet18

weights = models.ResNet18_Weights.DEFAULT
model_resnet_fe = models.resnet18(weights=weights)

# Freeze pretrained weights
for param in model_resnet_fe.parameters():
    param.requires_grad = False

# Replace the final layer for the three classes
model_resnet_fe.fc = nn.Linear(
    model_resnet_fe.fc.in_features,
    NUM_CLASSES
)

model_resnet_fe = model_resnet_fe.to(DEVICE)

# Optimize only the new final layer
criterion_fe = nn.CrossEntropyLoss()
optimizer_fe = torch.optim.Adam(
    model_resnet_fe.fc.parameters(),
    lr=0.001
)

# Training
start_time = time.time()

history_fe = train_model(
    model_resnet_fe,
    train_loader_resnet,
    val_loader_resnet,
    criterion_fe,
    optimizer_fe,
    EPOCHS,
    checkpoint_path="best_resnet_fe.pth"
)

training_time_fe = time.time() - start_time

print(f"Training time : {training_time_fe:.2f} secondes")
print(f"Best validation accuracy : {max(history_fe['val_acc']):.4f}")

## ResNet18 — Feature Extraction Setup

The pretrained convolutional layers are frozen. Only the replacement final classification layer is trained to predict the three parking-space categories.

## 26. Confusion Matrix — ResNet18 Feature Extraction (Test)

In [ ]:
model_resnet_fe.load_state_dict(
    torch.load("best_resnet_fe.pth", map_location=DEVICE)
)

model_resnet_fe.eval()

y_true_fe = []
y_pred_fe = []

with torch.no_grad():
    for images, labels in test_loader_resnet:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model_resnet_fe(images)
        predictions = outputs.argmax(dim=1)

        y_true_fe.extend(labels.cpu().numpy())
        y_pred_fe.extend(predictions.cpu().numpy())

cm_fe = confusion_matrix(y_true_fe, y_pred_fe)

class_names = ["Occupied", "Available", "Available but restricted"]

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_fe,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("ResNet18 Feature Extraction - Test")
plt.show()

print(classification_report(
    y_true_fe,
    y_pred_fe,
    target_names=class_names
))

## ResNet18 Feature Extraction — Test Results

- **Overall accuracy:** 77.71% (136 correct predictions out of 175 test images).

- **Occupied spaces:** 91% recall. The model recognizes most spaces containing a vehicle.

- **Available spaces:** 64% recall.

- **Available but restricted spaces:** 80% recall.

- **Main confusion:** 22 available spaces are predicted as available but restricted.

**Conclusion:** feature extraction provides a strong baseline for transfer learning, but the model still confuses the two unoccupied categories.

## 27. Transfer Learning — ResNet18 Fine-Tuning (Unfreeze `layer4`)

In [ ]:
# Load a new pretrained ResNet18
weights = models.ResNet18_Weights.DEFAULT
model_resnet_ft = models.resnet18(weights=weights)

# Freeze all parameters
for param in model_resnet_ft.parameters():
    param.requires_grad = False

# Replace the final layer for the three classes
model_resnet_ft.fc = nn.Linear(
    model_resnet_ft.fc.in_features,
    NUM_CLASSES
)

# Unfreeze only the final convolutional block
for param in model_resnet_ft.layer4.parameters():
    param.requires_grad = True

# Move the model to the selected device
model_resnet_ft = model_resnet_ft.to(DEVICE)

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in model_resnet_ft.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_resnet_ft.parameters()
)

print(f"Paramètres entraînables : {trainable_params:,}")
print(f"Paramètres totaux : {total_params:,}")

## ResNet18 — Fine-Tuning Setup

- **Pretrained model:** loads ResNet18 weights learned from ImageNet.

- **Freeze parameters:** initially freezes all pretrained parameters so they are not updated during training.

- **Final layer:** replaces the original classifier with a new layer for the three parking-space categories.

- **Unfreeze `layer4`:** allows the final convolutional block to adapt its visual features to parking-space images.

- **Device:** moves the model to the available GPU or CPU.

- **Trainable parameters:** counts the parameters that can be updated and compares them with the total number of parameters.

**Why fine-tuning?** Unlike feature extraction, fine-tuning updates part of the pretrained feature extractor as well as the new classifier. This allows the model to adapt its learned visual features to the parking-space task.

## 28. Configure the Loss Function and Optimizer — Fine-Tuning

In [ ]:
# Loss function
criterion_ft = nn.CrossEntropyLoss()

# Optimizer: trainable parameters only
optimizer_ft = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_resnet_ft.parameters()),
    lr=0.0001
)

# Set the number of epochs
EPOCHS = 20

## 29. Train ResNet18 with Fine-Tuning

In [ ]:
# Measure training time
start_time = time.time()

# Train ResNet18 with fine-tuning
history_ft = train_model(
    model_resnet_ft,
    train_loader_resnet,
    val_loader_resnet,
    criterion_ft,
    optimizer_ft,
    EPOCHS,
    checkpoint_path="best_resnet_ft.pth"
)

# Calculate elapsed time
training_time_ft = time.time() - start_time

print(f"Training time : {training_time_ft:.2f} secondes")
print(f"Best validation accuracy : {max(history_ft['val_acc']):.4f}")

## ResNet18 Fine-Tuning — Training Results

- **Best validation accuracy:** 87.43%, reached at epochs 18 and 19.

- **Final training accuracy:** 99.88%.

- **Final validation accuracy:** 86.29%.

- **Training loss:** decreases from 0.6660 to 0.0040, showing that the model fits the training data very closely.

- **Overfitting:** training accuracy approaches 100%, while validation accuracy remains around 87%. This gap suggests overfitting.

- **Training time:** 70.83 seconds for 20 epochs.

**Conclusion:** fine-tuning gives the strongest validation performance so far. However, the very high training accuracy indicates that the model may be memorizing some training examples.

## 30. Confusion Matrix — ResNet18 Fine-Tuning (Test)

In [ ]:
# Load the best saved weights
model_resnet_ft.load_state_dict(
    torch.load("best_resnet_ft.pth", map_location=DEVICE)
)

model_resnet_ft.eval()

y_true_ft = []
y_pred_ft = []

with torch.no_grad():
    for images, labels in test_loader_resnet:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model_resnet_ft(images)
        predictions = outputs.argmax(dim=1)

        y_true_ft.extend(labels.cpu().numpy())
        y_pred_ft.extend(predictions.cpu().numpy())

# Confusion matrix
cm_ft = confusion_matrix(y_true_ft, y_pred_ft)

class_names = ["Occupied", "Available", "Available but restricted"]

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_ft,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("ResNet18 Fine-Tuning - Test")
plt.show()

print(classification_report(
    y_true_ft,
    y_pred_ft,
    target_names=class_names
))

## ResNet18 Fine-Tuning — Test Results

- **Overall accuracy:** 86.86% (152 correct predictions out of 175 test images).

- **Occupied spaces:** 98% recall. The model identifies almost all occupied spaces.

- **Available spaces:** 82% recall.

- **Available but restricted spaces:** 81% recall.

- **Main confusions:** 10 available spaces are predicted as restricted, and 11 restricted spaces are predicted as available.

**Conclusion:** ResNet18 with fine-tuning achieves the strongest test result among the evaluated models and performs more evenly across the three categories.

## 31. Compare Model Performance

In [ ]:
import pandas as pd

comparaison = pd.DataFrame({
    "Modèle": [
        "CNN classique",
        "CNN + Augmentation 1",
        "CNN + Augmentation 2",
        "ResNet18 Feature Extraction",
        "ResNet18 Fine-Tuning"
    ],
    "Accuracy test (%)": [
        74.29, 71.43, 77.14, 77.71, 86.86
    ],
    "Temps entraînement (s)": [
        20.31, 47.15, 30.44, 71.48, 70.83
    ]
})

display(comparaison)

## Model Comparison

- **Best model: ResNet18 Fine-Tuning.** It achieves the highest test accuracy at 86.86%, about 9 percentage points above ResNet18 feature extraction.

- **Baseline CNN:** reaches 74.29% test accuracy in 20.31 seconds. It is the fastest model, but its accuracy is lower than the ResNet18 models.

- **Augmentation Strategy 1:** reaches 71.43%, below the baseline CNN. This augmentation did not improve test performance in this experiment.

- **Augmentation Strategy 2:** reaches 77.14%, outperforming the baseline CNN and strategy 1.

- **ResNet18 Feature Extraction:** reaches 77.71%. The pretrained convolutional layers are kept frozen while the final classifier is trained.

- **ResNet18 Fine-Tuning:** reaches 86.86%. Unfreezing `layer4` allows part of the pretrained feature extractor to adapt to the parking-space task.

- **Training time:** the baseline CNN is fastest (20.31 seconds). Both ResNet18 approaches take about 71 seconds.

**Conclusion:** ResNet18 with fine-tuning provides the best balance of accuracy and training time for this experiment. It improves test accuracy substantially while requiring a moderate increase in training time compared with the baseline CNN.

## 32. Visualize Misclassified Test Images — ResNet18 Fine-Tuning

In [ ]:
import matplotlib.pyplot as plt

# Load the best model
model_resnet_ft.load_state_dict(
    torch.load("best_resnet_ft.pth", map_location=DEVICE)
)
model_resnet_ft.eval()

class_names = ["Occupied", "Available", "Available but restricted"]

erreurs = []
position = 0

# Iterate over test images
with torch.no_grad():
    for images, labels in test_loader_resnet:

        images = images.to(DEVICE)
        outputs = model_resnet_ft(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()

        labels = labels.numpy()

        for i in range(len(labels)):
            if predictions[i] != labels[i]:
                original_idx = test_idx[position]

                erreurs.append((
                    original_idx,
                    labels[i],
                    predictions[i]
                ))

            position += 1

# Display up to six errors
nombre_images = min(6, len(erreurs))

print(f"Total misclassified images : {len(erreurs)}")

fig, axes = plt.subplots(
    2, 3, figsize=(12, 8)
)
axes = axes.flatten()

for i in range(nombre_images):
    original_idx, vraie_classe, prediction = erreurs[i]

    path, label = dataset.samples[original_idx]
    image = dataset.loader(path)

    axes[i].imshow(image)
    axes[i].set_title(
        f"True class : {class_names[vraie_classe]}\n"
        f"Predicted class : {class_names[prediction]}"
    )
    axes[i].axis("off")

for i in range(nombre_images, 6):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

### Error Analysis and Limitations

The misclassified examples show that distinguishing an available space from an available but restricted space can be difficult. The two categories may look similar in a photograph, especially when signs or pavement markings are not clearly visible. Camera angle, shadows, and image quality can also affect predictions.

ResNet18 with fine-tuning achieves 86.86% test accuracy in this experiment. Additional examples of ambiguous parking situations may help improve performance, although the remaining errors may be difficult to eliminate completely.